In [ ]:
# Project root — edit for your environment.
PROJ_ROOT = "/tscc/projects/ps-renlab2/jhc103/degu-genome-assembly-proj"


**Configuration note:** this notebook verifies that contaminant contigs flagged by NCBI FCS-GX / FCS-adapter (run on the NCBI Galaxy portal) did not land in the final chromosome scaffolds. The input paths below mirror `config.sh` at the repo root — edit them for your own data. The FCS reports (`contamination_action.txt`, `adaptor_report.txt`) are read from this working directory.


In [ ]:
# Input paths — edit for your environment (mirror config.sh at the repo root).
HAPHIC_BUILD_DIR = f"{PROJ_ROOT}/output/outputs-from-haphic-alignment/references_hifiasm_male403_hifiHiCMode_041425_trail1_allChrom/04.build"
fasta_file = f"{HAPHIC_BUILD_DIR}/assembly_final.fasta"
agp_file    = f"{HAPHIC_BUILD_DIR}/out_JBAT_review_with_orig_name.agp"


## Goal of this notebook
Check if any of the contaminating contigs are found made it into the scaffold

In [ ]:
import pandas as pd

In [ ]:
fasta_file = f"{PROJ_ROOT}/output/outputs-from-haphic-alignment/references_hifiasm_male403_hifiHiCMode_041425_trail1_allChrom/04.build/assembly_final.fasta"

# Read all lines, keep only those starting with '>'
with open(fasta_file, 'r') as f:
    # headers = [line.strip() for line in f if line.startswith('>')]
    headers = [line[1:].strip() for line in f if line.startswith('>')]

# Convert to DataFrame (optional, if you need tabular structure)
df_headers = pd.DataFrame(headers, columns=['header'])
print(df_headers)

In [ ]:
df_contam = pd.read_csv("contamination_action.txt", sep="\t", skiprows=1, skipfooter=1, engine='python')
contam = df_contam["#seq_id"].str.replace("lcl|", "", regex=False)
# contam =["ptg000006l"]

contam

In [ ]:
df_scaff = pd.read_csv(f"{PROJ_ROOT}/output/outputs-from-haphic-alignment/references_hifiasm_male403_hifiHiCMode_041425_trail1_allChrom/04.build/out_JBAT_review_with_orig_name.agp",sep="\t",skiprows=2, names=["scaffold_name","start","end","contig_num","type","contig_name","fillter","length","type2"])
df_scaff = df_scaff[["scaffold_name","contig_name"]]
df_scaff = df_scaff[df_scaff["contig_name"]!="100"]
df_scaff

Get the id for the first 30 chromosome

In [ ]:
# df_headers = df_headers[0:30]
# df_headers = df_headers[0:288]
df_headers

Select for the contig_name within the chromosome

In [ ]:
df_scaff = df_scaff[df_scaff["scaffold_name"].isin(df_headers["header"])]
df_scaff

 See if any of the contam is within the contig_name 

In [ ]:
df_scaff["contam"]=df_scaff["contig_name"].isin(contam)

In [ ]:
df_scaff["contam"].value_counts()

In [ ]:
df_scaff[df_scaff["contig_name"].isin(contam)]